In [1]:
import sqlite3

base_path = "/home/jonas/PycharmProjects/tikzcodegenerator/evaluation/results"
model="detikzify_2_5_8b"

conn = sqlite3.connect(f"{base_path}/{model}/promptfoo-db/promptfoo.db")
cursor = conn.cursor()

# Tabellen anzeigen
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tabellen:")
for table in tables:
    print(table[0])

conn.close()

Tabellen:
__drizzle_migrations
datasets
evals
evals_to_datasets
evals_to_prompts
prompts
evals_to_tags
tags
eval_results
configs
spans
traces
model_audits
blob_assets
blob_references


In [2]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px


class PromptfooMetricsLoader:
    def __init__(self, base_path: str | Path):
        self.base_path = Path(base_path)

    def find_models(self) -> list[str]:
        return sorted(
            db_path.parent.parent.name
            for db_path in self.base_path.glob("*/promptfoo-db/promptfoo.db")
        )

    def load_model(self, model: str) -> pd.DataFrame:
        db_path = self.base_path / model / "promptfoo-db" / "promptfoo.db"

        if not db_path.exists():
            raise FileNotFoundError(f"DB nicht gefunden: {db_path}")

        with sqlite3.connect(db_path) as conn:
            conn.row_factory = sqlite3.Row

            rows = conn.execute("""
                SELECT id, test_idx, score, success, named_scores
                FROM eval_results
            """).fetchall()

        data = []

        for row in rows:
            named_scores = json.loads(row["named_scores"]) if row["named_scores"] else {}

            entry = {
                "model": model,
                "id": row["id"],
                "test_idx": row["test_idx"],
                "score": row["score"],
                "success": row["success"],
            }

            entry.update(named_scores)
            data.append(entry)

        return pd.DataFrame(data).rename(columns={"ms-ssim": "ms_ssim"})

    def load_models(self, models: list[str] | None = None) -> pd.DataFrame:
        if models is None:
            models = self.find_models()

        dfs = [self.load_model(model) for model in models]

        if not dfs:
            raise ValueError("Keine Modelle gefunden.")

        return pd.concat(dfs, ignore_index=True)

    @staticmethod
    def to_long(df: pd.DataFrame) -> pd.DataFrame:
        meta_cols = ["model", "id", "test_idx", "score", "success"]
        metric_cols = [col for col in df.columns if col not in meta_cols]

        df_long = df.melt(
            id_vars=meta_cols,
            value_vars=metric_cols,
            var_name="metric",
            value_name="value",
        )

        df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
        return df_long.dropna(subset=["value"])


def plot_metrics(df_long: pd.DataFrame, title: str = "Promptfoo Metrics by Model"):
    fig = px.box(
        df_long,
        x="model",
        y="value",
        color="model",
        points="all",
        facet_col="metric",
        facet_col_wrap=4,
        title=title,
    )

    fig.update_xaxes(matches=None)
    fig.update_yaxes(matches=None)

    fig.update_layout(
        height=1400,
        showlegend=False,
    )

    fig.show()


def plot_score_metrics(df_long: pd.DataFrame):
    length_metrics = {"output_length", "reference_length"}

    df_scores = df_long[~df_long["metric"].isin(length_metrics)]

    plot_metrics(
        df_scores,
        title="Promptfoo Score Metrics by Model",
    )


def plot_length_metrics(df_long: pd.DataFrame):
    length_metrics = {"output_length", "reference_length"}

    df_lengths = df_long[df_long["metric"].isin(length_metrics)]

    plot_metrics(
        df_lengths,
        title="Promptfoo Length Metrics by Model",
    )


# -------------------------------------------------------------------
# Usage
# -------------------------------------------------------------------

base_path = "/home/jonas/PycharmProjects/tikzcodegenerator/evaluation/results"

loader = PromptfooMetricsLoader(base_path)

# Automatisch alle Modelle finden
df = loader.load_models()

# Alternativ bestimmte Modelle manuell laden:
# df = loader.load_models([
#     "detikzify_2_5_8b",
#     "anderes_model",
# ])

df_long = loader.to_long(df)

plot_score_metrics(df_long)
plot_length_metrics(df_long)